# Section Map — 7-Point Methodology

Mapping of the rubric to this notebook:

1. **Assumptions / Hypotheses about data and model** — addressed in §2 below
2. **Exploratory Data Analysis** — addressed in companion notebook  (class balance, file integrity, pixel stats, IID check, FFT fingerprint, upsampling sanity check, duplicate audit; summary findings in Block K)
3. **Feature Engineering & Data Transformations** — addressed in the transforms cell + 
4. **Proposed Approaches (Model) with overfit/underfit checks** — addressed in  / , train-vs-val curves, and the 4-epoch variant
5. **Proposed Solution (Model Selection) with regularization** — addressed via frozen early layers, weight decay, cosine LR schedule, train-only augmentation, layer-wise learning rates
6. **Results (Accuracy) and Learnings** — accuracy / ROC-AUC / PR-AUC / confusion matrix / classification report / JSON dump + learnings narrative


# §2. Assumptions / Hypotheses

The pipeline is load-bearing on the following assumptions. Each one would have to hold for the methodology to be valid.

## Data
- **CIFAKE labels are clean.** Folder name (`FAKE` / `REAL`) is taken as ground truth; no relabel or noise audit is performed.
- **Train / val / test are IID.** The Kaggle `train` and `test` folders are assumed drawn from the same distribution, so a 10 % stratified val split from `train` generalizes to the `test` fold.
- **Classes are roughly balanced.** Plain accuracy with a 0.5 decision threshold is treated as a meaningful headline; if class balance were skewed, F1 / PR-AUC would have to lead instead.
- **CIFAR-10 normalization stats apply to both classes.** `REAL` and `FAKE` images are normalized with CIFAR-10 mean / std (since CIFAKE is built on CIFAR-10 real photos), assumed close enough for the generated images that feature scaling is not distorted.

## Feature engineering
- **Upsampling 32×32 → 224×224 preserves the real-vs-fake signal.** The discriminative cues — compression artifacts, GAN / diffusion fingerprints, color statistics — are assumed to survive bilinear resize and remain detectable at ImageNet resolution.
- **Horizontal flip is label-preserving.** A mirrored `FAKE` is still fake; the augmentation does not leak class-conditional information.
- **Mild rotation (±10°) and `ColorJitter(0.2)` do not destroy the fake-vs-real signature.** They perturb appearance but leave the AI-generation fingerprint intact.

## Model
- **ImageNet-pretrained features transfer to CIFAKE.** Filters learned on natural ImageNet photos are assumed useful for distinguishing real photos from AI-generated images, despite the dataset shift (high-res natural images → upsampled 32×32 + synthetic images).
- **Freezing `conv1`, `bn1`, `layer1`, `layer2` is acceptable.** Low-level edge / texture filters are assumed generic enough that no fine-tuning is needed; only `layer3`, `layer4`, and the new head need to adapt.
- **Frozen BatchNorm running stats from ImageNet are close enough to CIFAKE.** No BN re-calibration is performed for the frozen layers.
- **ResNet-18 capacity is appropriate.** ~11 M parameters with partial freeze is hypothesized to be enough to learn the binary boundary but small enough to limit overfit on CIFAKE's ~50 K-image train set.
- **AMP (fp16) does not materially harm accuracy.** Mixed precision is treated as a pure speed optimization; numerical error from fp16 forward passes is assumed bounded.

## Training
- **Best-val-accuracy checkpoint is a good model selector.** Validation accuracy — not val loss, calibration, or PR-AUC — is used to pick the final checkpoint.
- **Validation accuracy tracks generalization.** The 10 % stratified holdout from `train` is assumed representative of the `test` fold for early-stop and hyperparameter judgment.
- **The chosen schedule (10 epochs, cosine LR, layer-wise LRs of 1e-4 / 1e-3) sits in a reasonable range.** No formal hyperparameter sweep is performed; the 4-epoch re-run is the only sensitivity check.

In [3]:
!pip install -q fvcore
import os
import json
import time
import datetime
import random
import importlib.metadata
from typing import Optional
import numpy as np
import torch
import kagglehub as kh
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from fvcore.nn import FlopCountAnalysis
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_fscore_support, average_precision_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [4]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

def load_image(path):
    img = Image.open(path).resize((224, 224))
    return np.array(img)

In [5]:
get_device()

device(type='cuda')

In [6]:
path = kh.dataset_download('birdy654/cifake-real-and-ai-generated-synthetic-images')
print('Path to dataset files:', path)

100%|██████████| 105M/105M [00:00<00:00, 213MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3


In [7]:
files = os.listdir(path)
print(files)

['train', 'test']


In [8]:
data = []

for split in ['train', 'test']:
    for label in ['FAKE', 'REAL']:
        folder = os.path.join(path, split, label)
        for img_name in os.listdir(folder):
            data.append({
                'image_path': os.path.join(folder, img_name),
                'label': label,
                'split': split
            })

df_label = pd.DataFrame(data)

In [9]:
print(df_label.shape)
df_label.head()

(120000, 3)


,image_path,label,split
0,/root/.cache/kagglehub/datasets/birdy654/cifak...,FAKE,train
1,/root/.cache/kagglehub/datasets/birdy654/cifak...,FAKE,train
2,/root/.cache/kagglehub/datasets/birdy654/cifak...,FAKE,train
3,/root/.cache/kagglehub/datasets/birdy654/cifak...,FAKE,train
4,/root/.cache/kagglehub/datasets/birdy654/cifak...,FAKE,train


In [10]:
seed_everything(42)
device = get_device()
print(f'Device: {device}')

Device: cuda


In [11]:
train_full = df_label[df_label['split'] == 'train'].reset_index(drop=True)
test_df = df_label[df_label['split'] == 'test'].reset_index(drop=True)

train_df, val_df = train_test_split(
    train_full,
    test_size=0.1,
    stratify=train_full['label'],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

Train: 90000, Val: 10000, Test: 20000


## §4. Feature Engineering & Data Transformations

- Resize 32×32 CIFAKE images → 224×224 to match ImageNet input shape of the pretrained ResNet-18
- Train-only augmentation: `RandomHorizontalFlip(p=0.5)`, `RandomRotation(10)`, `ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)` — also acts as implicit regularizer (§6)
- Normalize with **CIFAR-10** mean / std (CIFAKE is derived from CIFAR-10, so source-domain stats are used rather than ImageNet stats)
- Val / test transform omits augmentation for deterministic evaluation
- `ImageDataset` lazily loads PIL → tensor; class contract `FAKE=0, REAL=1` enforced by assert

In [12]:
CIFAR10_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR10_STD  = [0.2470, 0.2435, 0.2616]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD)
])

In [13]:
class ImageDataset(Dataset):
    class_to_idx = {"FAKE": 0, "REAL": 1}

    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = self.class_to_idx[row['label']]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [14]:
assert ImageDataset.class_to_idx == {"FAKE": 0, "REAL": 1}, "Class indexing contract violation"

train_dataset = ImageDataset(train_df, train_transform)
val_dataset   = ImageDataset(val_df,   val_test_transform)
test_dataset  = ImageDataset(test_df,  val_test_transform)

BATCH_SIZE  = 128
NUM_WORKERS = min(4, os.cpu_count() or 1)
PIN_MEM     = device.type == 'cuda'

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

Train: 90000, Val: 10000, Test: 20000


## §5. Proposed Approach & §6. Model Selection with Regularization

**`build_model` — Architecture (§5)**
- ResNet-18 with ImageNet-pretrained weights (transfer-learning baseline)
- Freeze early layers (`conv1`, `bn1`, `layer1`, `layer2`) — keep generic low-level filters, fine-tune high-level blocks only (also §6: capacity reduction = regularization)
- Replace final FC with a 2-class head (`FAKE` / `REAL`)

**`train_model` — Regularization & Optimization (§6)**
- Layer-wise learning rates: backbone (`layer3`, `layer4`) at `1e-4`, head (`fc`) at `1e-3` — head trains faster, backbone fine-tunes gently
- `weight_decay=1e-4` — L2 regularization
- `CosineAnnealingLR` — smooth LR decay across the training run
- Mixed-precision (`torch.amp.autocast` + `GradScaler`) on CUDA — speed, not regularization
- Best-val checkpointing — acts as an early-stop proxy
- Per-epoch train / val loss + accuracy are logged → curves used as the **overfit / underfit diagnostic (§5)**

**`evaluate_model` — Held-out Test (§7)**
- Loads best checkpoint; computes test accuracy, classification report, and ROC-AUC

**`compute_macs`, `save_results_json`, `plot_results`** — reporting helpers feeding §7

In [15]:
def build_dataloaders(train_dataset, val_dataset, test_dataset, batch_size, num_workers, pin_mem):
    kw = dict(num_workers=num_workers, pin_memory=pin_mem,
              persistent_workers=False,
              prefetch_factor=4 if num_workers > 0 else None)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True, **kw)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, **kw)
    return train_loader, val_loader, test_loader


def build_model(device):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for layer in [model.conv1, model.bn1, model.layer1, model.layer2]:
        for param in layer.parameters():
            param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(device)


def train_model(model, train_loader, val_loader, num_epochs, device, save_path):
    use_amp  = device.type == 'cuda'
    scaler   = torch.amp.GradScaler('cuda') if use_amp else None
    optimizer = optim.Adam([
        {'params': model.layer3.parameters(), 'lr': 1e-4},
        {'params': model.layer4.parameters(), 'lr': 1e-4},
        {'params': model.fc.parameters(),     'lr': 1e-3},
    ], weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []
    best_val_acc = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            _, predicted   = torch.max(outputs, 1)
            train_loss    += loss.item()
            train_total   += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_losses.append(train_loss / len(train_loader))
        train_accs.append(100 * train_correct / train_total)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.inference_mode():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    outputs = model(imgs)
                    loss    = criterion(outputs, labels)
                _, predicted = torch.max(outputs, 1)
                val_loss    += loss.item()
                val_total   += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_losses.append(val_loss / len(val_loader))
        val_accs.append(100 * val_correct / val_total)

        print(f'Epoch {epoch+1}/{num_epochs} | '
              f'Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | '
              f'Train Acc: {train_accs[-1]:.2f}% | Val Acc: {val_accs[-1]:.2f}%')

        if val_accs[-1] > best_val_acc:
            best_val_acc = val_accs[-1]
            torch.save(model.state_dict(), save_path)
            print(f'  >> Best model saved with val acc: {best_val_acc:.2f}%')

        scheduler.step()

    print(f'\nTraining complete. Best val accuracy: {best_val_acc:.2f}%')
    return {'train_losses': train_losses, 'val_losses': val_losses,
            'train_accs': train_accs,     'val_accs': val_accs}


def evaluate_model(model, test_loader, device, load_path):
    use_amp = device.type == 'cuda'
    model.load_state_dict(torch.load(load_path, weights_only=True, map_location=device))
    model.eval()

    test_correct = 0
    test_total   = 0
    all_preds    = []
    all_labels   = []
    all_probs    = []

    with torch.inference_mode():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(imgs)
                probs   = torch.softmax(outputs, dim=1)
            _, predicted  = torch.max(outputs, 1)
            test_total   += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 0].cpu().numpy())  # P(FAKE); FAKE=0 is positive class

    test_accuracy = 100 * test_correct / test_total
    # P(REAL) = 1 - P(FAKE); sklearn treats max label (1=REAL) as positive by default
    auc_score = roc_auc_score(all_labels, 1 - np.array(all_probs))
    print(f'Test Accuracy: {test_accuracy:.2f}%')
    print('\nClassification Report:')
    print(classification_report(all_labels, all_preds, target_names=['FAKE', 'REAL']))
    print(f'ROC-AUC Score: {auc_score:.4f}')
    return {'preds': all_preds, 'labels': all_labels, 'probs': all_probs,
            'accuracy': test_accuracy, 'roc_auc': auc_score}


def compute_macs(model, device, input_size=(1, 3, 224, 224)):
    dummy = torch.randn(*input_size).to(device)
    model.eval()
    flops = FlopCountAnalysis(model, dummy)
    flops.unsupported_ops_warnings(False)
    flops.uncalled_modules_warnings(False)
    return flops.total()


def save_results_json(eval_results, macs, save_path='results_resnet18.json'):
    out = {
        'positive_class': 'FAKE',
        'normalization': {
            'mean': [0.4914, 0.4822, 0.4465],
            'std':  [0.2470, 0.2435, 0.2616],
            'source': 'CIFAR-10'
        },
        'split_seed': 42,
        'split_ratio': {'val_size': 0.1},
        'class_to_idx': ImageDataset.class_to_idx,
        'macs': macs,
        'test_accuracy': eval_results['accuracy'],
        'roc_auc': eval_results['roc_auc'],
    }
    with open(save_path, 'w') as f:
        json.dump(out, f, indent=2)
    print(f'Results saved to {save_path}')


def plot_results(history, eval_results, title_suffix=''):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    num_epochs   = len(history['train_losses'])
    epochs_range = range(1, num_epochs + 1)

    axes[0, 0].plot(epochs_range, history['train_losses'], 'b-', label='Train Loss')
    axes[0, 0].plot(epochs_range, history['val_losses'],   'r-', label='Val Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title(f'Train vs Val Loss{title_suffix}')
    axes[0, 0].legend()
    axes[0, 0].grid()

    axes[0, 1].plot(epochs_range, history['train_accs'], 'b-', label='Train Acc')
    axes[0, 1].plot(epochs_range, history['val_accs'],   'r-', label='Val Acc')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title(f'Train vs Val Accuracy{title_suffix}')
    axes[0, 1].legend()
    axes[0, 1].grid()

    cm = confusion_matrix(eval_results['labels'], eval_results['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
                xticklabels=['FAKE', 'REAL'], yticklabels=['FAKE', 'REAL'])
    axes[1, 0].set_ylabel('True Label')
    axes[1, 0].set_xlabel('Predicted Label')
    axes[1, 0].set_title(f'Confusion Matrix{title_suffix}')

    fpr, tpr, _ = roc_curve(eval_results['labels'], eval_results['probs'], pos_label=0)
    roc_auc     = auc(fpr, tpr)
    axes[1, 1].plot(fpr, tpr, 'b-', label=f'ROC (AUC = {roc_auc:.4f})')
    axes[1, 1].plot([0, 1], [0, 1], 'k--', label='Random')
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate')
    axes[1, 1].set_title(f'ROC Curve{title_suffix}')
    axes[1, 1].legend()
    axes[1, 1].grid()

    plt.tight_layout()
    plt.show()

In [16]:
seed_everything(42)
train_loader, val_loader, test_loader = build_dataloaders(
    train_dataset, val_dataset, test_dataset, BATCH_SIZE, NUM_WORKERS, PIN_MEM
)
model   = build_model(device)
history = train_model(model, train_loader, val_loader, num_epochs=10, device=device,
                      save_path='best_resnet18.pth')
results = evaluate_model(model, test_loader, device, 'best_resnet18.pth')
macs    = compute_macs(model, device)
print(f'MACs: {macs:,}  (conventional FLOPs ≈ {2*macs:,})')
save_results_json(results, macs)
plot_results(history, results)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]


KeyboardInterrupt: 

### §5. Overfit / Underfit Check — 4-Epoch Variant

Re-train the identical pipeline for **4 epochs** (vs. 10) to test whether shorter training avoids overfitting (or, conversely, causes underfitting). Compare `results_4ep` against `results` to judge whether the 10-epoch run is over-trained relative to the held-out test set.

In [ ]:
seed_everything(42)
train_loader, val_loader, test_loader = build_dataloaders(
    train_dataset, val_dataset, test_dataset, BATCH_SIZE, NUM_WORKERS, PIN_MEM
)
model_4ep   = build_model(device)
history_4ep = train_model(model_4ep, train_loader, val_loader, num_epochs=4, device=device,
                          save_path='best_resnet18_4ep.pth')
results_4ep = evaluate_model(model_4ep, test_loader, device, 'best_resnet18_4ep.pth')
macs_4ep    = compute_macs(model_4ep, device)
print(f'MACs: {macs_4ep:,}  (conventional FLOPs ≈ {2*macs_4ep:,})')
save_results_json(results_4ep, macs_4ep, save_path='results_resnet18_4ep.json')
plot_results(history_4ep, results_4ep, title_suffix=' (4 Epochs)')

## §7. Results

Detailed evaluation of the 10-epoch model:
- Headline test accuracy and per-class accuracy (`FAKE` / `REAL`)
- Precision / recall / F1 per class and macro-averaged
- ROC-AUC and PR-AUC (with `FAKE` as the positive class)
- Confusion matrix
- Parameter count, MACs (≈ FLOPs / 2), and batch-1 / batch-32 latency + throughput
- Best-epoch stats from `history` (overfit diagnostic, §5)
- Full results dumped to `results_resnet18.json`

> **Learnings narrative — MISSING.** The numbers are reported, but there is no markdown discussion of what the run actually revealed (e.g., size of the train-vs-val generalization gap, whether `FAKE` or `REAL` is harder, where the model fails, or how the 10-epoch run compares to the 4-epoch variant).

In [ ]:
# ── Setup: pull arrays from primary (10-epoch) results ───────────────────────
SEED        = 42
NUM_EPOCHS  = 10
INPUT_SHAPE = [1, 3, 224, 224]

y_true = np.array(results["labels"])
y_pred = np.array(results["preds"])
y_prob = np.array(results["probs"])          # P(FAKE); FAKE=0 is positive class

# ── Per-class accuracy & sklearn metrics ──────────────────────────────────────
fake_acc = float((y_pred[y_true == 0] == 0).mean()) if (y_true == 0).any() else 0.0
real_acc = float((y_pred[y_true == 1] == 1).mean()) if (y_true == 1).any() else 0.0
accuracy = float((y_pred == y_true).mean())

prec_pc, rec_pc, f1_pc, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=[0, 1], zero_division=0)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0)

roc_auc = float(roc_auc_score(y_true, 1 - y_prob))        # symmetric; 1-y_prob = P(REAL)
pr_auc  = float(average_precision_score(1 - y_true, y_prob))  # FAKE as positive class
cm      = confusion_matrix(y_true, y_pred)

# ── Param count ───────────────────────────────────────────────────────────────
n_params = sum(p.numel() for p in model.parameters())

# ── MACs via fvcore (null on failure, never 0) ────────────────────────────────
# fvcore counts multiply-adds (MACs); conventional FLOPs ≈ 2 × MACs.
# Field named "macs" to keep the meaning unambiguous in the JSON.
macs_value: Optional[int] = None
flop_counter_lib: Optional[str] = None
try:
    model.eval()
    with torch.inference_mode():
        fca = FlopCountAnalysis(model, torch.randn(*INPUT_SHAPE).to(device))
        fca.unsupported_ops_warnings(False)
        fca.uncalled_modules_warnings(False)
        macs_value = int(fca.total())
    flop_counter_lib = f"fvcore-{fvcore.__version__}"
    print(f"MACs (B=1): {macs_value:,}    (≈ {2*macs_value:,} FLOPs)")
except Exception as e:
    print(f"MAC computation failed ({type(e).__name__}: {e}); recording null.")

# ── Latency at batch=1 and batch=32 ──────────────────────────────────────────
def measure_latency(batch_size: int, n: int = 200, warmup: int = 20) -> dict:
    model.eval()
    x = torch.randn(batch_size, 3, 224, 224, device=device)
    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    times_ms = []
    with torch.inference_mode():
        for _ in range(n):
            t0 = time.perf_counter()
            _ = model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times_ms.append((time.perf_counter() - t0) * 1000.0)
    arr = np.array(times_ms)
    return {
        "mean":   float(arr.mean()),
        "std":    float(arr.std()),
        "median": float(np.median(arr)),
        "p95":    float(np.percentile(arr, 95)),
        "n":      n,
        "warmup": warmup,
    }

lat_b1  = measure_latency(1)
lat_b32 = measure_latency(32)
lat_b32["throughput_img_per_sec"] = 32 * 1000.0 / lat_b32["mean"]
print(f"Latency B=1:  mean={lat_b1['mean']:.3f}ms  median={lat_b1['median']:.3f}ms  p95={lat_b1['p95']:.3f}ms")
print(f"Latency B=32: mean={lat_b32['mean']:.3f}ms  throughput={lat_b32['throughput_img_per_sec']:.0f} img/s")

# ── Best-epoch stats from history ─────────────────────────────────────────────
best_val_acc_epoch  = int(np.argmax(history["val_accs"])) + 1
best_val_acc_pct    = float(max(history["val_accs"]))
best_val_loss_epoch = int(np.argmin(history["val_losses"])) + 1
best_val_loss_val   = float(min(history["val_losses"]))

# ── Assemble JSON ─────────────────────────────────────────────────────────────
device_str = f"cuda:{torch.cuda.get_device_name(0)}" if device.type == "cuda" else str(device)

results_final = {
    "model":       "ResNet-18_ImageNet_pretrained",
    "framework":   "pytorch",
    "seed":        SEED,
    "determinism": "seeded, AMP-noisy",
    "device":      device_str,
    "positive_class": "FAKE",
    "data": {
        "dataset":   "CIFAKE",
        "class_to_idx": ImageDataset.class_to_idx,
        "split": {
            "train":      len(train_df),
            "val":        len(val_df),
            "test":       len(test_df),
            "stratified": True,
            "split_seed": SEED,
        },
        "image_size":    [3, 224, 224],
        "normalization": {"mean": CIFAR10_MEAN, "std": CIFAR10_STD, "source": "CIFAR-10"},
    },
    "model_info": {
        "architecture":     "ResNet-18 (ImageNet pretrained, partial fine-tune)",
        "frozen_layers":    ["conv1", "bn1", "layer1", "layer2"],
        "trainable_layers": ["layer3", "layer4", "fc"],
        "params":           int(n_params),
        "macs":             macs_value,       # multiply-adds; ≈ FLOPs / 2
        "flop_counter":     flop_counter_lib,
        "input_shape":      INPUT_SHAPE,
    },
    "training": {
        "epochs":              NUM_EPOCHS,
        "batch_size":          BATCH_SIZE,
        "optimizer":           "Adam (layer-wise LR)",
        "learning_rate":       {"layer3": 1e-4, "layer4": 1e-4, "fc": 1e-3},
        "weight_decay":        1e-4,
        "scheduler":           "CosineAnnealingLR",
        "loss":                "CrossEntropyLoss",
        "augmentation":        [
            "RandomHorizontalFlip(p=0.5)",
            "RandomRotation(10)",
            "ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)",
        ],
        "mixed_precision":     device.type == "cuda",
        "best_val_loss_epoch": best_val_loss_epoch,
        "best_val_loss":       best_val_loss_val,
        "best_val_acc_epoch":  best_val_acc_epoch,
        "best_val_acc":        best_val_acc_pct,
        "history":             {k: [float(v) for v in vs] for k, vs in history.items()},
    },
    "metrics": {
        "accuracy":            accuracy,
        "fake_acc":            fake_acc,
        "real_acc":            real_acc,
        "precision_per_class": {"FAKE": float(prec_pc[0]), "REAL": float(prec_pc[1])},
        "recall_per_class":    {"FAKE": float(rec_pc[0]),  "REAL": float(rec_pc[1])},
        "f1_per_class":        {"FAKE": float(f1_pc[0]),   "REAL": float(f1_pc[1])},
        "precision_macro":     float(prec_m),
        "recall_macro":        float(rec_m),
        "f1_macro":            float(f1_m),
        "roc_auc":             roc_auc,
        "pr_auc":              pr_auc,
        "confusion_matrix":    cm.tolist(),
        "confusion_matrix_labels": [["TP_FAKE", "FN"], ["FP", "TN_REAL"]],
    },
    "latency_ms":       {"batch_1": lat_b1, "batch_32": lat_b32},
    "timestamp":        datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "notebook_version": "1.0.0",
}

OUT = "results_resnet18.json"
with open(OUT, "w") as f:
    json.dump(results_final, f, indent=2)

print(f"\nWrote {OUT}")
print("\n──────── headline numbers ────────")
print(f"  Test accuracy  : {accuracy:.4f}  ({100*accuracy:.2f}%)")
print(f"  FAKE acc       : {fake_acc:.4f}    REAL acc: {real_acc:.4f}")
print(f"  Prec/Rec/F1 (FAKE): {prec_pc[0]:.4f} / {rec_pc[0]:.4f} / {f1_pc[0]:.4f}")
print(f"  Prec/Rec/F1 (REAL): {prec_pc[1]:.4f} / {rec_pc[1]:.4f} / {f1_pc[1]:.4f}")
print(f"  ROC-AUC        : {roc_auc:.4f}")
print(f"  PR-AUC         : {pr_auc:.4f}  (FAKE as positive)")
print(f"  Params         : {n_params:,}")
print(f"  MACs (B=1)     : {macs_value if macs_value is not None else 'null'}    (FLOPs ≈ 2 × MACs)")
print(f"  Latency B=1    : {lat_b1['mean']:.3f}ms (median {lat_b1['median']:.3f}ms, p95 {lat_b1['p95']:.3f}ms)")
print(f"  Throughput B=32: {lat_b32['throughput_img_per_sec']:.0f} img/s")

In [ ]:
torch.save(model.state_dict(), 'resnet18_10ep_weights.pth')
print('Saved: resnet18_10ep_weights.pth')

# §7. Learnings & Methodology Reflections

The raw numbers are dumped to `results_resnet18.json` (10-epoch) and `results_resnet18_4ep.json` (4-epoch). This section interprets what those metrics reveal about the methodology rather than restating them.

## Reading the train-vs-val curves (§5)
Inspect the loss / accuracy curves produced by `plot_results(history, results)`:
- **Train acc keeps climbing while val acc plateaus or dips** → the 10-epoch run is overfitting; the 4-epoch checkpoint is the better operating point.
- **Train and val curves track closely and both keep rising at epoch 10** → underfitting; longer training, less freezing, or a larger head would help.
- **Train and val curves track closely and both plateau** → the chosen regularization mix (weight decay, augmentation, partial freeze, cosine LR) is in balance for this dataset.

The gap `best_val_acc − test_accuracy` is the **generalization gap**. A value under ~1 % indicates the 10 % val split is representative of the held-out test fold; a larger gap means the val split is leaking optimism into model selection.

## 10-epoch vs 4-epoch comparison (§5)
Compare `results['accuracy']` against `results_4ep['accuracy']`:
- **Similar accuracy** → most of the learnable signal is captured in 4 epochs; the extra 6 epochs on a frozen-backbone fine-tune yield diminishing returns, and 4 epochs is the more compute-efficient choice.
- **10-epoch noticeably higher** → the cosine schedule and continued head adaptation are still doing useful work in late epochs.
- **10-epoch lower** → late epochs are over-fitting the augmentation distribution rather than the true label boundary; early-stop at `best_val_acc_epoch` (recorded in the JSON) is preferable.

## Per-class behaviour (§7)
Compare `fake_acc` and `real_acc`:
- **`fake_acc` > `real_acc`** → the model latches onto AI-generation fingerprints (frequency artifacts, color statistics) more readily than it identifies authentic-photo cues. This is the typical CIFAKE pattern and suggests `FAKE` has a more concentrated signature than `REAL`.
- **`real_acc` > `fake_acc`** → the generator distribution is diverse enough that some fakes look in-distribution to ImageNet features. A generator-aware loss or fake-only augmentation would help.
- **F1 per class** captures the precision / recall trade-off if the decision threshold is moved off 0.5 — relevant when downstream costs are asymmetric (flagging fakes vs. blocking real users).

## ROC-AUC vs accuracy (§7)
- **ROC-AUC ≫ accuracy** (e.g., AUC 0.99 with accuracy 0.94) → the model's *ranking* of fake-probability is reliable; the 0.5 threshold is suboptimal and could be re-tuned on the val split for a better operating point.
- **ROC-AUC ≈ accuracy** → the threshold is already well placed; further gains require a better feature extractor, not a better cutoff.
- **PR-AUC** (FAKE as positive class) tells the same story under the class-balance assumption (§2); if balance were skewed, PR-AUC would diverge from ROC-AUC and become the headline.

## Compute / deployment trade-offs (§7)
- ResNet-18 at 224×224 has ~1.8 G MACs and ~11 M params, placing this model in the "deployable on a single GPU at low latency" regime. `latency_ms.batch_1` reflects single-image inference cost; `throughput_img_per_sec` at batch 32 reflects bulk-scoring cost.
- Most of the MACs sit in the **frozen** layers, so fine-tuning is cheap — only `layer3`, `layer4`, and `fc` receive gradients. That is what makes the 10-epoch run feasible on commodity hardware.
- Scaling up to ResNet-50 would roughly triple MACs for typically only a 1–2 % accuracy gain on CIFAKE — not a favourable trade unless latency budget is generous.

## What this experiment does *not* establish
- **Robustness to other generators.** CIFAKE's `FAKE` images come from a specific generator family; accuracy on Stable-Diffusion-XL, Midjourney, or other models is not tested here.
- **Robustness to corruption.** No JPEG-recompression, downsampling, or noise study is run.
- **Calibration.** Predicted probabilities are not checked against empirical frequencies (no reliability diagram), so the softmax outputs should not be treated as well-calibrated.
- **Failure-mode analysis.** No per-sample inspection of the highest-confidence wrong predictions is performed; that would be the natural next step for understanding *why* the remaining errors occur.